# Lab 7: Thực hành chuyên sâu về Phân tích
cú pháp phụ thuộc (Dependency Parsing)

## Phần 1: Giới thiệu và cài đặt 

In [1]:
!pip install -U spacy


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!python -m spacy download en_core_web_md

     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta 


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Phần 2: Phân tích câu và Trực quan hóa 

### Tải mô hình và phân tích câu 

In [3]:
import spacy
from spacy import displacy

# Tải mô hình tiếng Anh đã cài đặt
# Sử dụng en_core_web_md vì nó chứa các vector từ và cây cú pháp đầy đủ
nlp = spacy.load("en_core_web_md")
# Câu ví dụ
text = "The quick brown fox jumps over the lazy dog."
# Phân tích câu với pipeline của spaCy
doc = nlp(text)

### 2.2 Trực quan hóa cây phụ thuộc 

In [4]:
displacy.serve(doc, style="dep")

d:\NLP_DL\env\Lib\site-packages\spacy\displacy\__init__.py:108: UserWarning: [W011] It looks like you're calling displacy.serve from within a Jupyter notebook or a similar environment. This likely means you're already running a local web server, so there's no need to make displaCy start another one. Instead, you should be able to replace displacy.serve with displacy.render to show the visualization.
  warnings.warn(Warnings.W011)



Using the 'dep' visualizer
Serving on http://0.0.0.0:5000 ...

Shutting down server on port 5000.


- Từ gốc (ROOT) của câu là: `jumps` (VERB). Động từ chính, biểu thị hành động của câu. 
- `jumps` có các từ phụ thuộc (dependent) là: 
  - `fox` -> quan hệ **nsubj** (nomial subject - chủ ngữ). `fox` là chủ ngữ của động từ `jumps`.
  - `over` -> quan hệ **prep** (prepositional modifier - giới từ). `over` là giới từ chỉ quan hệ nơi chốn liên kết với động từ `jumps`
- `fox` là head của các từ: 
  - `The` -> quan hệ **det** (mạo từ xác định cho danh từ `fox`)
  - `quick` và `brown` -> quan hệ **amod** (tính từ bổ nghĩa, mô tả đặc điểm của `fox`)


## Phần 3: Truy cập các thành phần trong cây phụ thuộc

In [ ]:
# Lấy một câu khác để phân tích
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = nlp(text)
# In ra thông tin của từng token
print(f"{'TEXT':<12} | {'DEP':<10} | {'HEAD TEXT':<12} | {'HEAD POS':<8} | {'CHILDREN'}")
print("-" * 70)
for token in doc:
  # Trích xuất các thuộc tính
  children = [child.text for child in token.children]
  print(f"{token.text:<12} | {token.dep_:<10} | {token.head.text:<12} | {token.head.pos_:<8} | {children}")

TEXT         | DEP        | HEAD TEXT    | HEAD POS | CHILDREN
----------------------------------------------------------------------
Apple        | nsubj      | looking      | VERB     | []
is           | aux        | looking      | VERB     | []
looking      | ROOT       | looking      | VERB     | ['Apple', 'is', 'at']
at           | prep       | looking      | VERB     | ['buying']
buying       | pcomp      | at           | ADP      | ['startup']
U.K.         | compound   | startup      | NOUN     | []
startup      | dobj       | buying       | VERB     | ['U.K.', 'for']
for          | prep       | startup      | NOUN     | ['billion']
$            | quantmod   | billion      | NUM      | []
1            | compound   | billion      | NUM      | []
billion      | pobj       | for          | ADP      | ['$', '1']


## Phần 4: Duyệt cây phụ thuộc để trích xuất thông tin 

### 4.1 Bài toán: Tìm chủ ngữ và tân ngữ của một động từ 

In [ ]:
text = "The cat chased the mouse and the dog watched them."
doc = nlp(text)

for token in doc: 
  if token.pos_ == "VERB":
    verb = token.text
    subject = ""
    obj = ""

    for child in token.children:
      if child.dep_ == "nsubj":
        subject = child.text
      if child.dep_ == "dobj":
        obj = child.text
    if subject and obj:
      print(f"Found Triplet: ({subject}, {verb}, {obj})")

Found Triplet: (cat, chased, mouse)
Found Triplet: (dog, watched, them)


### 4.2 Bài toán: Tìm các tính từ bổ nghĩa cho một danh từ 

In [ ]:
text = "The big, fluffy white cat is sleeping on the warm mat."
doc = nlp(text)

for token in doc:
  # Chỉ tìm các danh từ
  if token.pos_ == "NOUN":
    adjectives = []
  # Tìm các tính từ bổ nghĩa (amod) trong các con của danh từ
    for child in token.children:
      if child.dep_ == "amod":
        adjectives.append(child.text)
    if adjectives:
      print(f"Danh từ '{token.text}' được bổ nghĩa bởi các tính từ: {adjectives}")

Danh từ 'cat' được bổ nghĩa bởi các tính từ: ['big', 'fluffy', 'white']
Danh từ 'mat' được bổ nghĩa bởi các tính từ: ['warm']


## Phần 5: Bài tập tự luyện 

### Bài 1: Tìm động từ chính của câu 

In [ ]:
def find_main_verb(doc):
  for token in doc:
    if token.dep_ == "ROOT" and token.pos_ == "VERB":
      return token.text
  return None

In [ ]:
sentence = "The dog barked loudly at the stranger."
doc = nlp(sentence)

main_verb = find_main_verb(doc)
print(f"Động từ chính của câu là: {main_verb}")

Động từ chính của câu là: barked


### Bài 2: Trích xuất các cụm danh từ (Noun Chunks)

In [5]:
def extract_noun_chunks(doc):
  noun_chunks = []
  for token in doc: 
    if token.pos_ == "NOUN":
      chunk_tokens = [token.text]
      for child in token.children:
        if child.dep_ in ("amod", "det", "compound"):
          chunk_tokens.append(child.text)
      chunk_tokens.sort(key=lambda x: doc.text.index(x))
      noun_chunk = " ".join(chunk_tokens)
      noun_chunks.append(noun_chunk)
  return noun_chunks

In [6]:
sentence = "The beautiful garden behind the old house has many colorful flowers."
doc = nlp(sentence)
noun_chunks = extract_noun_chunks(doc)
print("Noun Chunks found in the sentence:")
for chunk in noun_chunks:
    print(chunk)

Noun Chunks found in the sentence:
The beautiful garden
the old house
many colorful flowers


### Bài 3: Tìm đường đi ngắn nhất trong cây 

In [ ]:
def get_path_to_root(token):
    path = [token]
    while token.head != token:
        token = token.head
        path.append(token)
    return path

In [19]:
sentence = "The teacher gave the students difficult homework yesterday."
doc = nlp(sentence)

token = [t for t in doc if t.text == "difficult"][0]
path = get_path_to_root(token)

print("Đường đi từ '{}' lên ROOT:".format(token.text))
print(" → ".join([t.text for t in path]))

Đường đi từ 'difficult' lên ROOT:
difficult → homework → gave
